In [2]:
import pandas as pd

df = pd.read_csv("corporate_income_statement.csv")

print("Dataset shape (rows, columns):", df.shape)
print("\nColumn list:")
print(df.columns.tolist())

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

df.head()

Dataset shape (rows, columns): (1750, 27)

Column list:
['symbol', 'fiscalDateEnding', 'reportedCurrency', 'grossProfit', 'totalRevenue', 'costOfRevenue', 'costofGoodsAndServicesSold', 'operatingIncome', 'sellingGeneralAndAdministrative', 'researchAndDevelopment', 'operatingExpenses', 'investmentIncomeNet', 'netInterestIncome', 'interestIncome', 'interestExpense', 'nonInterestIncome', 'otherNonOperatingIncome', 'depreciation', 'depreciationAndAmortization', 'incomeBeforeTax', 'incomeTaxExpense', 'interestAndDebtExpense', 'netIncomeFromContinuingOperations', 'comprehensiveIncomeNetOfTax', 'ebit', 'ebitda', 'netIncome']

Missing values per column:
symbol                                  0
fiscalDateEnding                        0
reportedCurrency                        2
grossProfit                             4
totalRevenue                            0
costOfRevenue                           1
costofGoodsAndServicesSold              1
operatingIncome                         4
sellingGen

,symbol,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,...,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome
0,NVDA,2025-10-31,USD,4.184900e+10,5.700600e+10,1.515700e+10,1.515700e+10,3.601000e+10,1.134000e+09,4.705000e+09,...,NaN,751000000.0,3.793600e+10,6.026000e+09,NaN,3.191000e+10,NaN,3.799700e+10,3.874800e+10,3.191000e+10
1,NVDA,2025-07-31,USD,3.385300e+10,4.674300e+10,1.289000e+10,1.289000e+10,2.844000e+10,1.122000e+09,4.291000e+09,...,NaN,669000000.0,3.120600e+10,4.784000e+09,NaN,2.642200e+10,NaN,3.126800e+10,3.193700e+10,2.642200e+10
2,NVDA,2025-04-30,USD,2.666800e+10,4.406200e+10,1.739400e+10,1.739400e+10,2.163800e+10,1.041000e+09,3.989000e+09,...,NaN,611000000.0,2.191000e+10,3.135000e+09,NaN,1.877500e+10,NaN,2.197300e+10,2.258400e+10,1.877500e+10
3,NVDA,2025-01-31,USD,2.872300e+10,3.933100e+10,1.060800e+10,1.060800e+10,2.403400e+10,9.750000e+08,3.714000e+09,...,NaN,543000000.0,2.521700e+10,3.126000e+09,NaN,2.209100e+10,NaN,2.527800e+10,2.582100e+10,2.209100e+10
4,NVDA,2024-10-31,USD,2.615600e+10,3.508200e+10,8.926000e+09,8.926000e+09,2.186900e+10,8.970000e+08,3.390000e+09,...,NaN,478000000.0,2.231600e+10,3.007000e+09,NaN,1.930900e+10,NaN,2.237700e+10,2.285500e+10,1.930900e+10


In [3]:
import pandas as pd

# 1. Remove exact duplicate rows
df_clean = df.drop_duplicates().copy()

# 2. Drop columns with massive missingness (e.g., 100% or >60% missing)
cols_to_drop = ['investmentIncomeNet', 'netInterestIncome']
df_clean = df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns])

# 3. Convert fiscalDateEnding to datetime and extract year
df_clean['fiscalDateEnding'] = pd.to_datetime(df_clean['fiscalDateEnding'], errors='coerce')
df_clean['fiscal_year'] = df_clean['fiscalDateEnding'].dt.year

# 4. Convert key financial columns to numeric
numeric_cols = [
    'grossProfit', 'totalRevenue', 'costOfRevenue',
    'costofGoodsAndServicesSold', 'operatingIncome',
    'sellingGeneralAndAdministrative', 'researchAndDevelopment',
    'operatingExpenses'
]
for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# 5. Drop rows missing core metrics (grossProfit / operatingIncome)
df_clean = df_clean.dropna(subset=['grossProfit', 'operatingIncome'])

# 6. Check cleaned shape and remaining missing values
print("Cleaned shape (rows, columns):", df_clean.shape)
print("\nRemaining missing values (top 15):")
print(df_clean.isnull().sum().sort_values(ascending=False).head(15))

Cleaned shape (rows, columns): (1746, 26)

Remaining missing values (top 15):
depreciation                         1746
nonInterestIncome                    1746
interestAndDebtExpense               1746
comprehensiveIncomeNetOfTax          1746
interestIncome                       1194
otherNonOperatingIncome               943
researchAndDevelopment                320
netIncomeFromContinuingOperations     162
sellingGeneralAndAdministrative        64
depreciationAndAmortization            24
interestExpense                        21
ebitda                                 13
reportedCurrency                        2
costOfRevenue                           1
costofGoodsAndServicesSold              1
dtype: int64


In [4]:
import numpy as np
import pandas as pd

# 1. Drop columns with 100% missing values or >50% missingness
cols_to_drop = [
    'depreciation',
    'nonInterestIncome',
    'interestAndDebtExpense',
    'comprehensiveIncomeNetOfTax',
    'interestIncome',
    'otherNonOperatingIncome'
]
df_ready = df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns]).copy()

# 2. Fill missing reportedCurrency with mode (or 'UNKNOWN')
if 'reportedCurrency' in df_ready.columns:
    mode_curr = df_ready['reportedCurrency'].mode()
    fill_curr = mode_curr[0] if not mode_curr.empty else 'UNKNOWN'
    df_ready['reportedCurrency'] = df_ready['reportedCurrency'].fillna(fill_curr)

# 3. Feature engineering: key financial ratios (safe division)
df_ready['gross_margin_pct'] = np.where(
    df_ready['totalRevenue'] != 0,
    (df_ready['grossProfit'] / df_ready['totalRevenue']) * 100,
    np.nan
).round(2)

df_ready['operating_margin_pct'] = np.where(
    df_ready['totalRevenue'] != 0,
    (df_ready['operatingIncome'] / df_ready['totalRevenue']) * 100,
    np.nan
).round(2)

df_ready['opex_ratio_pct'] = np.where(
    (df_ready['totalRevenue'] != 0) & ('operatingExpenses' in df_ready.columns),
    (df_ready['operatingExpenses'] / df_ready['totalRevenue']) * 100,
    np.nan
).round(2)

# 4. Save cleaned dataset
output_filename = "corporate_income_statement_clean.csv"
df_ready.to_csv(output_filename, index=False)

print("Final cleaned shape (rows, columns):", df_ready.shape)
print("Saved file:", output_filename)
print("\nSample engineered metrics:")
print(df_ready[['symbol', 'fiscal_year', 'totalRevenue', 'gross_margin_pct', 'operating_margin_pct']].head())

Final cleaned shape (rows, columns): (1746, 23)
Saved file: corporate_income_statement_clean.csv

Sample engineered metrics:
  symbol  fiscal_year  totalRevenue  gross_margin_pct  operating_margin_pct
0   NVDA         2025  5.700600e+10             73.41                 63.17
1   NVDA         2025  4.674300e+10             72.42                 60.84
2   NVDA         2025  4.406200e+10             60.52                 49.11
3   NVDA         2025  3.933100e+10             73.03                 61.11
4   NVDA         2024  3.508200e+10             74.56                 62.34
